# BoxBunny Runner

Essential commands for building, running, and testing the boxing robot system.

**Quick Start:** Run cells 1-4 in order, then launch what you need.

---
## 1. Build & Setup
Build the ROS 2 workspace and seed demo data.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash

echo "=== Building Workspace ==="
colcon build --symlink-install 2>&1
echo ""
source install/setup.bash
echo "=== Packages ==="
ros2 pkg list 2>/dev/null | grep boxbunny || echo "(build failed)"
echo ""
echo "=== Seeding Demo Data ==="
python3 tools/demo_data_seeder.py

---
## 2. Run Tests

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
python3 -m pytest tests/ -v --tb=short 2>&1

---
## 3. System Check
Hardware, dependencies, and model status in one view.

In [ ]:
%run notebooks/scripts/hardware_check.py

---
## 4. Launch System
Starts all ROS nodes + IMU simulator + GUI.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Launching BoxBunny in dev mode..."
ros2 launch boxbunny_core boxbunny_dev.launch.py &
LAUNCH_PID=$!
echo "Launch PID: $LAUNCH_PID"
echo $LAUNCH_PID > /tmp/boxbunny_launch.pid
sleep 5
echo ""
echo "=== Active ROS Nodes ==="
ros2 node list 2>/dev/null || echo "(nodes still starting...)"
echo ""
echo "Run 'Stop System' cell to shut down" 

---
## 5. Stop System

In [ ]:
%%bash
set +e
if [ -f /tmp/boxbunny_launch.pid ]; then
    PID=$(cat /tmp/boxbunny_launch.pid)
    kill $PID 2>/dev/null && echo "Stopped launch process $PID" || echo "Process already stopped"
fi
pkill -f 'boxbunny_core' 2>/dev/null
pkill -f 'boxbunny_gui' 2>/dev/null
pkill -f 'boxbunny_dashboard' 2>/dev/null
pkill -f 'imu_simulator' 2>/dev/null
echo "All BoxBunny processes stopped" 

---
## 6. Phone Dashboard
Starts the dashboard server with a public tunnel URL and QR code.
Scan the QR with your phone to open the dashboard from anywhere.

In [ ]:
%run notebooks/scripts/dashboard_tunnel.py

---
## 7. CV Model Live Test
Opens a camera window showing pose skeleton + predicted action labels.
Stand 1.5-2m from the camera and throw punches. Press **q** to stop.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
python3 action_prediction/run.py \
    --checkpoint action_prediction/model/best_model.pth \
    --pose-weights action_prediction/model/yolo26n-pose.pt \
    2>&1 || echo "(D435i camera not connected or models missing)" 

---
## 8. LLM Coach Test
Loads the local Qwen 2.5-3B model and generates a sample coaching tip.

In [ ]:
%run notebooks/scripts/test_llm.py

---
## 9. Build Vue Frontend
Rebuild the phone dashboard SPA after making frontend changes.

In [ ]:
%%bash
set +e
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/src/boxbunny_dashboard/frontend
npm run build 2>&1

---
## 10. Sound Test
Play all sound effects to verify audio.

In [ ]:
%run notebooks/scripts/play_sounds.py

---
## 11. Demo User Profiles
Visual cards for each demo user with XP, streaks, and records.

In [ ]:
%run notebooks/scripts/view_user_dashboards.py

---
## 12. Benchmark Test
Percentile rankings for demo users against population norms.

In [ ]:
%run notebooks/scripts/benchmark_test.py